In [ ]:
import os

# change to ur home path
OUT_PATH = "/home/jovyan/transaxx/ext_modules/include/nn/cuda/axx_mults/mul8s_FPGA_ISH2.h" 

def m1(a, b):
    return 7 if (a == 3 and b == 3) else a * b

def m3(a, b):
    return 11 if (a == 3 and b == 3) else a * b

def m5(a, b):
    return a * b

M = {"1": m1, "3": m3, "5": m5}

def r4(code, a, b):
    ah = (a >> 2) & 0b11
    al = a & 0b11
    bh = (b >> 2) & 0b11
    bl = b & 0b11

    return (
        (M[code[0]](ah, bh) << 4) +
        (M[code[1]](ah, bl) << 2) +
        (M[code[2]](al, bh) << 2) +
        M[code[3]](al, bl)
    )

def ish2_u8(a, b):
    ah = (a >> 4) & 0xF
    al = a & 0xF
    bh = (b >> 4) & 0xF
    bl = b & 0xF
# Output8x8 =
# 256(AH × BH)
# + 16(AH × BL)
# + 16(AL × BH)
# + AL × BL
# AH×BH uses R1315
# AH×BL uses R1311
# AL×BH uses R1311
# AL×BL uses R1311
    return (
        (r4("1315", ah, bh) << 8) +
        (r4("1311", ah, bl) << 4) +
        (r4("1311", al, bh) << 4) +
        r4("1311", al, bl)
    )

def to_s8(x):
    return x - 256 if x >= 128 else x

def approx_s8(a_raw, b_raw):
    a = to_s8(a_raw)
    b = to_s8(b_raw)

    sign = -1 if (a < 0) ^ (b < 0) else 1
    return sign * ish2_u8(abs(a), abs(b))

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

with open(OUT_PATH, "w") as f:
    f.write("#include <stdint.h>\n\n")
    f.write("const __device__ int16_t lut[256][256] = {")

    for a in range(256):
        row = [str(approx_s8(a, b)) for b in range(256)]
        f.write(("{" if a == 0 else ",\n{") + ", ".join(row) + "}")

    f.write("};\n")

print("Saved:", OUT_PATH)

Saved: /home/jovyan/transaxx/ext_modules/include/nn/cuda/axx_mults/mul8s_FPGA_ISH2.h
